# 35 — Comparing FFT similarity/difference metrics by position

**Goal.** For a single box we record the same object at several positions and get one complex
FFT per position (shape `(1, L, F, 2)`: L lasers, F freqs, 2 = x/y). A *good* box is one where
the FFT changes a lot as the position changes, so positions are easy to tell apart from their FFTs.

Notebook `34_ncc.ipynb` scores a box by the **off-diagonal sum of NCC** (complex cosine similarity):
low sum -> FFTs are dissimilar -> easy to distinguish -> good box. Here we ask: *is NCC-sum the best
score?* We compute a suite of pairwise metrics, turn each into a box score, and compare them.

### Two ideas to keep in mind

1. **Similarity vs distance.** Some metrics are similarities (high = alike: NCC, magnitude-cosine,
   Pearson, coherence) and some are distances (high = different: spectral angle, Euclidean, KL, JS).
   For a *good* box we want low similarity / high distance.

2. **Average vs worst-case (bottleneck).** "Can I distinguish *every* position?" is a worst-case
   property: two positions are confusable if *any* pair is too similar, no matter the average. So for
   each metric we report both the mean over pairs **and** the bottleneck pair (max similarity / min
   distance). The bottleneck is usually the more honest box score for distinguishability.

In [1]:
import sys
sys.path.append('/Users/eitanturok/good-vibrations/src3')

import json
from pathlib import Path
from itertools import combinations

import numpy as np
import pandas as pd

In [2]:
BOX = 'cardboard'
SPEAKER = 2  # only keep samples recorded with this speaker
DROP_EMPTY_BOX = False  # keep the empty-box pair; its com_dist is set to 200 (very far)

BASE_SAMPLE_DIR = Path(rf'D:\eturok\experiment-20\data\{BOX}\samples')

In [3]:
def load_metadata(sample_dir):
    """Merge the single-key dicts in metadata.jsonl into one dict."""
    meta = {}
    with open(sample_dir / 'metadata.jsonl') as f:
        for line in f:
            line = line.strip()
            if line:
                meta.update(json.loads(line))
    return meta


rows = []
for sample_dir in sorted(BASE_SAMPLE_DIR.iterdir()):
    if not sample_dir.is_dir():
        continue
    meta = load_metadata(sample_dir)
    rows.append({
        'sample_id': meta['sample_id'],
        'speaker': meta['speaker'],
        'com': np.array(meta['com'], dtype=float),
        'fft_path': sample_dir / 'inputs' / '03_fft_shifts.npz',
    })

df = pd.DataFrame(rows)
df = df[df['speaker'] == SPEAKER]
if DROP_EMPTY_BOX:
    df = df[~df['com'].apply(lambda c: np.all(c == -1))]
order = np.lexsort((df['com'].apply(lambda c: c[1]).to_numpy(),
                    df['com'].apply(lambda c: c[0]).to_numpy()))
df = df.iloc[order].reset_index(drop=True)
df

,sample_id,speaker,com,fft_path
0,000073,2,"[-1.0, -1.0]",D:\eturok\experiment-20\data\cardboard\samples...
1,000065,2,"[72.5762987012987, 134.17694805194805]",D:\eturok\experiment-20\data\cardboard\samples...
2,000017,2,"[73.36605657237936, 182.27787021630616]",D:\eturok\experiment-20\data\cardboard\samples...
3,000041,2,"[74.52380952380952, 157.4055829228243]",D:\eturok\experiment-20\data\cardboard\samples...
4,000057,2,"[93.28410914927768, 132.93258426966293]",D:\eturok\experiment-20\data\cardboard\samples...
5,000009,2,"[94.10726072607261, 181.04620462046205]",D:\eturok\experiment-20\data\cardboard\samples...
6,000033,2,"[94.29256198347107, 156.5586776859504]",D:\eturok\experiment-20\data\cardboard\samples...
7,000001,2,"[113.23205342237061, 178.35559265442404]",D:\eturok\experiment-20\data\cardboard\samples...
8,000025,2,"[113.29883138564274, 154.2220367278798]",D:\eturok\experiment-20\data\cardboard\samples...
9,000049,2,"[114.24592833876221, 131.77524429967426]",D:\eturok\experiment-20\data\cardboard\samples...


## Metric suite

Each metric takes two FFT arrays `(1, L, F, 2)` and returns a scalar. We flatten the complex array to
one long vector for the vector-based metrics; for coherence we keep the per-laser/per-direction channels
so we have an ensemble to average over.

| metric | kind | uses phase? | what it captures |
|---|---|---|---|
| `ncc` | sim | **yes** | complex cosine similarity (notebook 34's metric) |
| `mag_cosine` | sim | no | cosine similarity of magnitude spectra only |
| `pearson_mag` | sim | no | linear correlation of magnitude spectra (mean-centered) |
| `coherence` | sim | **yes** | mean over freq of channel-ensemble magnitude-squared coherence |
| `spectral_angle` | dist | no | angle between magnitude spectra = arccos(mag_cosine) |
| `euclid_mag` | dist | no | Euclidean distance of L2-normalized magnitude spectra |
| `sym_kl` | dist | no | symmetrized KL of magnitude-as-distribution |
| `js` | dist | no | Jensen-Shannon divergence (bounded, symmetric KL cousin) |

**A note on coherence.** Textbook magnitude-squared coherence needs averaging over an ensemble of
segments; on a single FFT realization it is identically 1 and useless. Here we get an ensemble for
free: the `L*2` laser/direction channels. So `coherence(f)` is a per-frequency NCC across channels,
averaged over frequency — a frequency-resolved cousin of `ncc`.

**A note on KL / JS.** These treat the (normalized) magnitude spectrum as a probability distribution,
so they are sensitive to the *shape* of the spectrum but blind to its overall scale and to phase.
JS is the better-behaved of the two (symmetric and bounded in `[0, ln 2]`).

In [4]:
def _flat_complex(fft):
    return np.asarray(fft, dtype=np.complex128).ravel()

def _flat_mag(fft):
    return np.abs(_flat_complex(fft))

def _prob(fft, eps=1e-12):
    """Magnitude spectrum normalized to sum to 1, so it can be read as a distribution."""
    m = _flat_mag(fft) + eps
    return m / m.sum()


def ncc(a, b):
    """Complex cosine similarity |<a, b>| / (||a|| ||b||) in [0, 1]. Uses phase. (notebook 34)"""
    a, b = _flat_complex(a), _flat_complex(b)
    return float(np.abs(np.vdot(a, b)) / (np.linalg.norm(a) * np.linalg.norm(b)))

def mag_cosine(a, b):
    """Cosine similarity of magnitude spectra in [0, 1]. Phase discarded."""
    a, b = _flat_mag(a), _flat_mag(b)
    return float(np.dot(a, b) / (np.linalg.norm(a) * np.linalg.norm(b)))

def pearson_mag(a, b):
    """Pearson correlation of magnitude spectra in [-1, 1]."""
    return float(np.corrcoef(_flat_mag(a), _flat_mag(b))[0, 1])

def coherence(a, b):
    """Mean over freq of channel-ensemble magnitude-squared coherence in [0, 1]. Uses phase.

    The L lasers x 2 directions act as the ensemble that ordinary coherence needs. At each
    frequency f: |sum_k conj(A_k) B_k|^2 / (sum_k|A_k|^2 sum_k|B_k|^2), then averaged over f.
    """
    A = np.asarray(a, dtype=np.complex128)[0]  # (L, F, 2)
    B = np.asarray(b, dtype=np.complex128)[0]
    A = np.transpose(A, (0, 2, 1)).reshape(-1, A.shape[1])  # (K=L*2, F)
    B = np.transpose(B, (0, 2, 1)).reshape(-1, B.shape[1])
    cross = np.abs(np.sum(np.conj(A) * B, axis=0)) ** 2     # (F,)
    denom = np.sum(np.abs(A) ** 2, axis=0) * np.sum(np.abs(B) ** 2, axis=0)
    return float(np.mean(cross / denom))

def spectral_angle(a, b):
    """Spectral angle mapper: arccos of magnitude cosine, in [0, pi/2]. Distance."""
    return float(np.arccos(np.clip(mag_cosine(a, b), -1.0, 1.0)))

def euclid_mag(a, b):
    """Euclidean distance between L2-normalized magnitude spectra, in [0, sqrt(2)]. Distance."""
    a, b = _flat_mag(a), _flat_mag(b)
    a, b = a / np.linalg.norm(a), b / np.linalg.norm(b)
    return float(np.linalg.norm(a - b))

def sym_kl(a, b):
    """Symmetrized KL of magnitude-as-distribution. Distance >= 0."""
    p, q = _prob(a), _prob(b)
    return float(0.5 * np.sum(p * np.log(p / q)) + 0.5 * np.sum(q * np.log(q / p)))

def js(a, b):
    """Jensen-Shannon divergence in [0, ln 2]. Symmetric, bounded. Distance."""
    p, q = _prob(a), _prob(b)
    m = 0.5 * (p + q)
    return float(0.5 * np.sum(p * np.log(p / m)) + 0.5 * np.sum(q * np.log(q / m)))


# kind: 'sim' (low = good box) or 'dist' (high = good box)
METRICS = {
    'ncc':            ('sim',  ncc),
    'mag_cosine':     ('sim',  mag_cosine),
    'pearson_mag':    ('sim',  pearson_mag),
    'coherence':      ('sim',  coherence),
    'spectral_angle': ('dist', spectral_angle),
    'euclid_mag':     ('dist', euclid_mag),
    'sym_kl':         ('dist', sym_kl),
    'js':             ('dist', js),
}

## Compute every metric for every position pair

In [5]:
def load_fft(path):
    with np.load(path) as d:
        return d['fft']

# Load each FFT once (loading dominates the cost).
ffts = {row.sample_id: load_fft(row.fft_path) for row in df.itertuples()}
coms = {row.sample_id: row.com for row in df.itertuples()}

pair_rows = []
for sid1, sid2 in combinations(df['sample_id'], 2):
    a, b = ffts[sid1], ffts[sid2]
    row = {
        'sample_id1': sid1, 'sample_id2': sid2,
        # Empty-box pair (com == [-1, -1]) has no real position: treat it as very far.
        'com_dist': (200.0 if np.any(coms[sid1] == -1) or np.any(coms[sid2] == -1)
                     else float(np.linalg.norm(coms[sid1] - coms[sid2]))),
    }
    for name, (_kind, fn) in METRICS.items():
        row[name] = fn(a, b)
    pair_rows.append(row)

pair_df = pd.DataFrame(pair_rows)
pair_df

,sample_id1,sample_id2,com_dist,ncc,mag_cosine,pearson_mag,coherence,spectral_angle,euclid_mag,sym_kl,js
0,000073,000065,200.000000,0.904059,0.999284,0.999077,0.955098,0.037844,0.037842,0.003637,0.000887
1,000073,000017,200.000000,0.985030,0.999205,0.998971,0.953569,0.039871,0.039869,0.003852,0.000939
2,000073,000041,200.000000,0.768919,0.999321,0.999123,0.954284,0.036862,0.036860,0.003636,0.000886
3,000073,000057,200.000000,0.956442,0.999069,0.998800,0.950956,0.043152,0.043149,0.004395,0.001072
4,000073,000009,200.000000,0.946070,0.999216,0.998984,0.953885,0.039607,0.039604,0.003807,0.000931
5,000073,000033,200.000000,0.955726,0.999118,0.998860,0.950053,0.042004,0.042001,0.004298,0.001051
6,000073,000001,200.000000,0.979130,0.999039,0.998758,0.953273,0.043840,0.043837,0.004048,0.000988
7,000073,000025,200.000000,0.950809,0.999108,0.998849,0.951385,0.042250,0.042247,0.004193,0.001025
8,000073,000049,200.000000,0.844021,0.999232,0.999006,0.954295,0.039190,0.039188,0.003829,0.000932
9,000065,000017,48.107405,0.953778,0.999113,0.998856,0.957445,0.042113,0.042110,0.003521,0.000860


## Per-metric box score

For each metric we summarize the box two ways:

- **mean** over all position pairs — the "average distinguishability" (this is what notebook 34's
  sum is, up to the constant number of pairs).
- **bottleneck** — the worst (most-confusable) pair: `max` for similarities, `min` for distances.
  This is the pair that limits how well you can separate *all* positions.

For both columns we also report a `*_norm` in `[0, 1]` where **higher = better box** (more
distinguishable), so the columns are directly comparable across metrics with different ranges. We
normalize each metric by its own min/max across the pairs, then flip similarities so that, for every
metric, 1 = most distinguishable.

In [6]:
def normalize01(x):
    x = np.asarray(x, dtype=float)
    lo, hi = np.nanmin(x), np.nanmax(x)
    return np.full_like(x, 0.5) if hi == lo else (x - lo) / (hi - lo)

summary_rows = []
for name, (kind, _fn) in METRICS.items():
    vals = pair_df[name].to_numpy()
    norm = normalize01(vals)
    if kind == 'sim':          # low sim = good -> flip so high = good
        norm = 1.0 - norm
        bottleneck_raw = vals.max()        # most-similar pair limits separability
    else:                       # high dist = good
        bottleneck_raw = vals.min()        # least-distant pair limits separability
    summary_rows.append({
        'metric': name,
        'kind': kind,
        'mean_raw': vals.mean(),
        'bottleneck_raw': bottleneck_raw,
        'mean_score': norm.mean(),         # higher = better box
        'bottleneck_score': norm.min(),    # higher = better box (worst pair, normalized)
    })

summary_df = pd.DataFrame(summary_rows).set_index('metric')
summary_df.round(4)

,kind,mean_raw,bottleneck_raw,mean_score,bottleneck_score
metric,,,,,
ncc,sim,0.9486,0.9988,0.2185,0.0
mag_cosine,sim,0.9991,0.9996,0.3392,0.0
pearson_mag,sim,0.9988,0.9995,0.3397,0.0
coherence,sim,0.9587,0.9637,0.3705,0.0
spectral_angle,dist,0.0421,0.0267,0.4184,0.0
euclid_mag,dist,0.0421,0.0267,0.4184,0.0
sym_kl,dist,0.0037,0.0021,0.3917,0.0
js,dist,0.0009,0.0005,0.3912,0.0


## Heatmaps: one matrix per metric

Same layout as notebook 34. Lets you see *where* each metric thinks positions are confusable, and
whether the different metrics agree on the hard pairs.

In [7]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

sample_ids = list(df['sample_id'])
n = len(sample_ids)
idx = {sid: i for i, sid in enumerate(sample_ids)}
tick = [f'{sid}<br>x={coms[sid][0]:.0f}<br>y={coms[sid][1]:.0f}' for sid in sample_ids]

def metric_matrix(name):
    M = np.full((n, n), np.nan)
    for r in pair_df.itertuples():
        i, j = idx[r.sample_id1], idx[r.sample_id2]
        v = getattr(r, name)
        M[i, j] = M[j, i] = v
    M[np.triu(np.ones((n, n), bool), k=1)] = np.nan  # keep lower triangle
    return M

names = list(METRICS)
ncols = 2
nrows = int(np.ceil(len(names) / ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=names,
                    horizontal_spacing=0.12, vertical_spacing=0.06)
for k, name in enumerate(names):
    r, c = k // ncols + 1, k % ncols + 1
    M = metric_matrix(name)
    rev = METRICS[name][0] == 'sim'  # similarities: reverse so dark = confusable
    fig.add_trace(go.Heatmap(z=M, colorscale='Blues', reversescale=rev,
                             showscale=False, hoverongaps=False), row=r, col=c)
    fig.update_yaxes(autorange='reversed', row=r, col=c)
fig.update_layout(height=320 * nrows, width=900,
                  title=f'Pairwise FFT metrics ({BOX} box, speaker {SPEAKER})')
fig.show()

## Do the metrics agree? (rank correlation across pairs)

If two metrics rank the position pairs the same way (which pairs are most similar), they carry the
same information and you only need one. We orient every metric as a **distance** (flip similarities)
and compute Spearman rank correlation across the pairs. Blocks of high correlation = redundant metrics.

In [8]:
# Orient all metrics as distances (high = different), then Spearman-correlate across pairs.
oriented = pd.DataFrame(index=pair_df.index)
for name, (kind, _fn) in METRICS.items():
    oriented[name] = -pair_df[name] if kind == 'sim' else pair_df[name]

corr = oriented.corr(method='spearman')

fig = go.Figure(go.Heatmap(
    z=corr.values, x=corr.columns, y=corr.index, zmin=-1, zmax=1, colorscale='RdBu',
    reversescale=True, text=corr.round(2).values, texttemplate='%{text}',
    colorbar=dict(title='Spearman')))
fig.update_layout(title='Agreement between metrics (Spearman rank corr across pairs)',
                  width=760, height=680)
fig.update_yaxes(autorange='reversed')
fig.show()
corr.round(3)

,ncc,mag_cosine,pearson_mag,coherence,spectral_angle,euclid_mag,sym_kl,js
ncc,1.000,-0.274,-0.271,0.037,-0.274,-0.274,-0.206,-0.214
mag_cosine,-0.274,1.000,1.000,0.565,1.000,1.000,0.905,0.915
pearson_mag,-0.271,1.000,1.000,0.566,1.000,1.000,0.906,0.916
coherence,0.037,0.565,0.566,1.000,0.565,0.565,0.741,0.735
spectral_angle,-0.274,1.000,1.000,0.565,1.000,1.000,0.905,0.915
euclid_mag,-0.274,1.000,1.000,0.565,1.000,1.000,0.905,0.915
sym_kl,-0.206,0.905,0.906,0.741,0.905,0.905,1.000,0.999
js,-0.214,0.915,0.916,0.735,0.915,0.915,0.999,1.000


## Does FFT difference track physical distance?

A reassuring sanity check (not the objective itself): if a metric is meaningful, position pairs that
are physically farther apart (larger COM distance) should also be farther apart in FFT space. We plot
each metric vs `com_dist` and report the Spearman correlation. A metric that is *flat* vs distance is
not picking up position at all; a metric with steep, monotone growth even at *small* `com_dist` is
exactly what we want (nearby positions already look very different).

In [9]:
from scipy.stats import spearmanr

fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=names,
                    horizontal_spacing=0.10, vertical_spacing=0.09)
for k, name in enumerate(names):
    r, c = k // ncols + 1, k % ncols + 1
    x = pair_df['com_dist'].to_numpy()
    y = pair_df[name].to_numpy()
    rho, _ = spearmanr(x, y)
    fig.add_trace(go.Scatter(x=x, y=y, mode='markers',
                             marker=dict(size=6, opacity=0.7), showlegend=False),
                  row=r, col=c)
    fig.layout.annotations[k].text = f'{name}  (rho={rho:.2f})'
    fig.update_xaxes(title_text='COM distance', row=r, col=c)
fig.update_layout(height=320 * nrows, width=900,
                  title='FFT metric vs physical COM distance')
fig.show()

## Takeaways & recommendation

Read off the notebook above, but the design guidance:

**Which metric.**
- **`ncc` and `coherence` use phase; the magnitude-only metrics (`mag_cosine`, `pearson_mag`,
  `spectral_angle`, `euclid_mag`, `js`, `sym_kl`) throw phase away.** Phase encodes how the box's
  vibration is delayed/shaped by the object, which is a lot of the position signal — so a phase-aware
  metric should win. Check the Spearman-agreement heatmap: if `ncc`/`coherence` separate from the
  magnitude cluster, phase is carrying real information and you should keep a phase-aware metric.
- **KL / JS** treat the spectrum as a probability distribution: they ignore overall vibration *scale*
  and phase, and need an epsilon hack for zeros. They answer "is the spectral *shape* different,"
  which is probably not your question. Use them only if you specifically want scale-invariance.
- **`pearson_mag`** removes a per-spectrum mean/scale, so it is the most forgiving (positions look
  similar); usually the *least* discriminative for this task.
- `spectral_angle` / `euclid_mag` are just monotone re-expressions of `mag_cosine` — expect rho ~ 1
  with it. Pick one, drop the rest.

**Which summary (this matters more than the metric).** Distinguishability is **worst-case**: a box is
only as separable as its most-confusable pair. Prefer the **bottleneck** column (max similarity / min
distance) over the mean/sum. The off-diagonal *sum* can be dragged up or down by many easy pairs while
one confusable pair quietly ruins classification — exactly the case the sum hides.

**The cleaner objective, if you can get repeats.** Ultimately you want "can a classifier read off the
position from the FFT." With only one sample per position you cannot estimate within-position noise,
so any single-number score is a proxy. If you record each position 2-3 times, switch to a
**silhouette score** (mean inter-position distance vs within-position scatter) or **nearest-neighbor
classification accuracy** in your chosen metric — those *are* the distinguishability you want to rank
boxes by, and they automatically reward the bottleneck.

**Suggested box score to start:** `1 - max_off_diag(ncc)` (phase-aware, worst-case). Compare its box
ranking against `bottleneck_score` for `coherence` and `euclid_mag` across several boxes; if they
disagree, the disagreement tells you whether phase / worst-case framing changes your conclusions.

## Quantifying the trend vs COM distance (the main question)

We want a metric where **greater COM distance => greater FFT difference**, ideally with a steep,
monotone trend. Two separate statistical questions:

1. **Is the trend monotone?** -> **Spearman rho**. Only cares about ordering, ignores curve shape.
   This is the headline number for "which metric tracks position best."
2. **Is it linear / proportional ("double COM => double metric"), or non-linear?** ->
   - **Pearson r**: strength of the *linear* part. If Spearman >> Pearson, the trend is monotone but
     **curved** (saturating).
   - **OLS** `diff ~ com_dist`: slope, intercept, R^2. Strict proportionality means intercept ~ 0.
   - **log-log slope** = exponent `p` in `diff ~ com_dist^p`. **p~1 linear**, p<1 saturating, p>1 superlinear.

Everything is computed on a **difference** version of each metric (>= 0, increasing with dissimilarity):
similarities become `1 - value`, distances are used as-is. That way "up = more different" for all of them,
and log-log is well defined.

**Expect the bounded similarities (ncc, coherence, mag_cosine, pearson_mag) to look concave** — they
saturate near 1, so they cannot keep growing with COM distance. The distance metrics have more room
to stay linear.

**Local sensitivity** (last column) is the mean difference among the *closest* third of position
pairs (smallest COM distance). For distinguishability this is what really matters: a good box already
produces big FFT differences for *small* moves, i.e. a steep slope near the origin.

In [10]:
import numpy as np
from scipy.stats import spearmanr, pearsonr, linregress

# Difference version of each metric: >= 0 and increasing with dissimilarity.
def to_diff(name, vals):
    kind = METRICS[name][0]
    return (1.0 - vals) if kind == 'sim' else vals

x = pair_df['com_dist'].to_numpy()
close_mask = x <= np.quantile(x, 1/3)  # closest third of pairs

trend_rows = []
for name in METRICS:
    d = to_diff(name, pair_df[name].to_numpy())

    rho = spearmanr(x, d).correlation        # monotone trend (headline)
    r = pearsonr(x, d)[0]                     # linear part
    lin = linregress(x, d)                    # OLS: slope/intercept/rvalue

    # log-log exponent p in  d ~ com_dist^p  (drop nonpositive d for the log)
    pos = d > 0
    if pos.sum() >= 3:
        ll = linregress(np.log(x[pos]), np.log(d[pos]))
        exponent = ll.slope
    else:
        exponent = np.nan

    trend_rows.append({
        'metric': name,
        'spearman': rho,                      # >0.7 good monotone trend
        'pearson': r,                         # compare to spearman: gap => curved
        'ols_slope': lin.slope,
        'ols_intercept': lin.intercept,       # ~0 => proportional (passes through origin)
        'ols_r2': lin.rvalue ** 2,
        'loglog_exp': exponent,               # ~1 linear, <1 saturating, >1 superlinear
        'local_sens': d[close_mask].mean(),   # mean diff among closest 1/3 of pairs
    })

trend_df = (pd.DataFrame(trend_rows)
            .set_index('metric')
            .sort_values('spearman', ascending=False))
trend_df.round(3)

,spearman,pearson,ols_slope,ols_intercept,ols_r2,loglog_exp,local_sens
metric,,,,,,,
coherence,0.763,0.866,0.0,0.038,0.750,0.089,0.039
sym_kl,0.263,0.154,0.0,0.004,0.024,0.069,0.004
js,0.262,0.147,0.0,0.001,0.022,0.068,0.001
ncc,0.204,0.241,0.0,0.037,0.058,0.539,0.045
mag_cosine,0.128,-0.089,-0.0,0.001,0.008,0.027,0.001
spectral_angle,0.128,-0.051,-0.0,0.043,0.003,0.014,0.040
euclid_mag,0.128,-0.051,-0.0,0.043,0.003,0.014,0.040
pearson_mag,0.126,-0.087,-0.0,0.001,0.008,0.028,0.001


### Scatter with fitted trend line per metric

Each panel plots the **difference** form of the metric vs COM distance, with the OLS line. Read off:
a steep, tight, upward line with points already high at small COM distance is the ideal. A flat or
very scattered panel means that metric barely senses position. Compare the slope near the left edge
(small COM distance) across metrics — that left-edge behavior is what governs distinguishability.

In [11]:
import plotly.graph_objects as go
from plotly.subplots import make_subplots

names = list(METRICS)
ncols = 2
nrows = int(np.ceil(len(names) / ncols))
fig = make_subplots(rows=nrows, cols=ncols, subplot_titles=names,
                    horizontal_spacing=0.10, vertical_spacing=0.09)
xs_line = np.linspace(x.min(), x.max(), 50)
for k, name in enumerate(names):
    r_, c_ = k // ncols + 1, k % ncols + 1
    d = to_diff(name, pair_df[name].to_numpy())
    lin = linregress(x, d)
    rho = spearmanr(x, d).correlation
    fig.add_trace(go.Scatter(x=x, y=d, mode='markers',
                             marker=dict(size=6, opacity=0.7), showlegend=False), row=r_, col=c_)
    fig.add_trace(go.Scatter(x=xs_line, y=lin.intercept + lin.slope * xs_line, mode='lines',
                             line=dict(color='crimson', width=2), showlegend=False), row=r_, col=c_)
    fig.layout.annotations[k].text = f'{name}  (rho={rho:.2f}, R^2={lin.rvalue**2:.2f})'
    fig.update_xaxes(title_text='COM distance', row=r_, col=c_)
    fig.update_yaxes(title_text='difference', row=r_, col=c_)
fig.update_layout(height=320 * nrows, width=900,
                  title='Metric difference vs COM distance, with OLS trend')
fig.show()

### How to read `trend_df` to pick a metric

- **Rank metrics by `spearman`** — that is "tracks position monotonically," your main criterion.
- **`pearson` vs `spearman` gap** tells you about shape: small gap => roughly linear; big gap =>
  monotone but **saturating** (the bounded similarities will show this).
- **`loglog_exp`** answers your "doubling" question directly: ~1 means double COM ~ double metric;
  noticeably <1 means it saturates (diminishing returns at large COM distance).
- **`ols_intercept` near 0** means the metric is roughly *proportional* to COM distance.
- **`local_sens`** is the tie-breaker that matches the real goal: among metrics with similar
  Spearman, prefer the one with the **largest** local sensitivity (biggest FFT difference for the
  closest position pairs).

One honest limitation to keep in mind: `com_dist` is a scalar summary of a 2D move, so part of the
scatter is real direction-dependence, not metric noise — don't expect R^2 near 1 even for a good metric.

## Across all speakers: which metric tracks position best on average?

Everything above used a single speaker. Each speaker excites the box differently, so a metric that
only works for one speaker is not trustworthy. Here we repeat the trend analysis (Spearman rho and
OLS R^2 of the metric difference vs `com_dist`) **for every speaker**, in two regimes:

- **with empty box** — includes the empty-box pairs at `com_dist = 200`. Mixes "detect present vs
  absent" with "localize among positions".
- **without empty box** — real positions only. Isolates **localization**, which is the harder,
  more meaningful signal for ranking boxes.

We then **average each stat over speakers** per metric. The metric with the highest *mean
`rho_without`* (consistent monotone localization across speakers) is the one to adopt.

In [12]:
# Re-scan every sample across ALL speakers (the df above is filtered to one speaker).
all_rows = []
for sample_dir in sorted(BASE_SAMPLE_DIR.iterdir()):
    if not sample_dir.is_dir():
        continue
    meta = load_metadata(sample_dir)
    all_rows.append({
        'sample_id': meta['sample_id'],
        'speaker': meta['speaker'],
        'com': np.array(meta['com'], dtype=float),
        'fft_path': sample_dir / 'inputs' / '03_fft_shifts.npz',
    })
df_all = pd.DataFrame(all_rows)
speakers = sorted(df_all['speaker'].unique())
print('speakers:', speakers)
df_all['speaker'].value_counts().sort_index()

speakers: [1, 2, 3, 4, 5, 6, 7, 8]


speaker
1    10
2    10
3    10
4    10
5    10
6    10
7    10
8    10
Name: count, dtype: int64

In [13]:
from scipy.stats import spearmanr, linregress

def is_empty(com):
    return bool(np.any(np.asarray(com) == -1))

def pairs_for_speaker(sub):
    """All position pairs for one speaker, with every metric + com_dist (empty pair -> 200)."""
    ffts_s = {r.sample_id: load_fft(r.fft_path) for r in sub.itertuples()}  # local: freed per speaker
    coms_s = {r.sample_id: r.com for r in sub.itertuples()}
    rows = []
    for sid1, sid2 in combinations(sub['sample_id'], 2):
        empty = is_empty(coms_s[sid1]) or is_empty(coms_s[sid2])
        cd = 200.0 if empty else float(np.linalg.norm(coms_s[sid1] - coms_s[sid2]))
        row = {'com_dist': cd, 'is_empty_pair': empty}
        for name, (_kind, fn) in METRICS.items():
            row[name] = fn(ffts_s[sid1], ffts_s[sid2])
        rows.append(row)
    return pd.DataFrame(rows)

def rho_r2(pdf, name):
    """Spearman rho and OLS R^2 of the metric's difference form vs com_dist."""
    if len(pdf) < 3:
        return np.nan, np.nan
    x = pdf['com_dist'].to_numpy()
    d = to_diff(name, pdf[name].to_numpy())
    return spearmanr(x, d).correlation, linregress(x, d).rvalue ** 2

In [17]:
# Per-speaker rho/R^2 for every metric, with and without the empty-box pairs.
records = []
for spk in speakers:
    sub = df_all[df_all['speaker'] == spk]
    if len(sub) < 3:
        print(f'skipping speaker {spk}: only {len(sub)} samples')
        continue
    pdf = pairs_for_speaker(sub)
    pdf_no = pdf[~pdf['is_empty_pair']]
    for name in METRICS:
        rho_w, r2_w = rho_r2(pdf, name)
        rho_o, r2_o = rho_r2(pdf_no, name)
        records.append({'speaker': spk, 'metric': name,
                        'rho_with': rho_w, 'r2_with': r2_w,
                        'rho_without': rho_o, 'r2_without': r2_o})

per_speaker = pd.DataFrame(records)
per_speaker.round(3).head(64)

KeyboardInterrupt: 

In [22]:
per_speaker.round(3).head(32)

,speaker,metric,rho_with,r2_with,rho_without,r2_without
0,1,ncc,0.113,0.003,-0.054,0.015
1,1,mag_cosine,0.495,0.004,0.478,0.098
2,1,pearson_mag,0.494,0.005,0.477,0.105
3,1,coherence,0.617,0.090,0.710,0.479
4,1,spectral_angle,0.495,0.013,0.478,0.118
5,1,euclid_mag,0.495,0.013,0.478,0.118
6,1,sym_kl,0.422,0.003,0.471,0.118
7,1,js,0.424,0.003,0.467,0.117
8,2,ncc,0.204,0.058,0.028,0.000
9,2,mag_cosine,0.128,0.008,0.244,0.062


In [23]:
per_speaker.to_csv(r'C:\Users\eitanturok\good-vibrations\notebooks\per_speaker.csv', index=False)


### Average across speakers (the ranking)

`*_mean` is the average over speakers; `rho_without_std` shows how *consistent* a metric is across
speakers (low = reliable). Sorted by `rho_without_mean` — your headline "best localization metric".

In [15]:
agg = per_speaker.groupby('metric').agg(
    rho_with_mean=('rho_with', 'mean'),
    r2_with_mean=('r2_with', 'mean'),
    rho_without_mean=('rho_without', 'mean'),
    rho_without_std=('rho_without', 'std'),
    r2_without_mean=('r2_without', 'mean'),
).sort_values('rho_without_mean', ascending=False)
agg.round(3)

,rho_with_mean,r2_with_mean,rho_without_mean,rho_without_std,r2_without_mean
metric,,,,,
js,0.458,0.037,0.514,0.139,0.237
sym_kl,0.456,0.038,0.512,0.145,0.234
pearson_mag,0.462,0.045,0.511,0.134,0.251
euclid_mag,0.461,0.052,0.510,0.131,0.261
mag_cosine,0.461,0.045,0.510,0.131,0.250
spectral_angle,0.461,0.052,0.510,0.131,0.261
coherence,0.484,0.204,0.484,0.198,0.248
ncc,-0.042,0.019,-0.016,0.133,0.028


In [16]:
import plotly.graph_objects as go

# Bar chart: mean Spearman rho across speakers per metric, with/without empty box,
# with std error bars on the "without" bars (consistency across speakers).
order = agg.index.tolist()
fig = go.Figure()
fig.add_bar(x=order, y=agg['rho_with_mean'], name='with empty box')
fig.add_bar(x=order, y=agg['rho_without_mean'], name='without empty box (localization)',
            error_y=dict(type='data', array=agg['rho_without_std'], visible=True))
fig.update_layout(
    title='Mean Spearman rho across speakers (metric difference vs COM distance)',
    xaxis_title='metric', yaxis_title='mean rho over speakers',
    barmode='group', width=900, height=500)
fig.show()

### Reading the result

- **Pick the metric with the highest `rho_without_mean`** — it monotonically tracks *position*
  (not just presence/absence) and does so consistently across the different speaker excitations.
- **`rho_without_std`** is the tie-breaker: prefer a metric that is reliable across speakers (low
  std) over one that is great for one speaker and useless for another.
- **Compare `rho_with_mean` vs `rho_without_mean`**: a big drop from with->without means the metric
  mostly detects the empty box rather than localizing real positions. A metric strong in *both*
  columns is the most useful.

If a single metric wins both columns with low std, that is your box score going forward. If `coherence`
keeps winning here as it did for the single speaker, adopt it in notebook 34 in place of NCC.